Project 3, Person A: preprocessing, the Apriori vs FP-Growth benchmark, and the AI audit

This notebook covers Task 3.1, Task 3.2, and the Part II AI exercise. The functions it calls live in common.py next to this notebook, so Person B can import the same encoded datasets. At the Nov 2 merge, common.py gets pasted into the Setup section of the final notebook so the submission runs on its own.

Code reuse. Loading Online_Retail.xlsx, the missing-value check, the cancellation test on InvoiceNo, and the Description whitespace cleanup come from our Assignment 1 notebook (assgn1/final.ipynb, cells 2 and 27). The Assignment 1 regex for collapsing spaces had a bug and is fixed in common.py. The constants-at-the-top layout, the plotting pattern, and the AI audit layout (unedited AI code in a raw cell, then the audit, then the fixed code) come from Assignment 2 (assgn2/final_submission.ipynb cells 2 and 24, and assgn2/Ai_integration.ipynb). Each reused piece is also marked in a comment where it appears.

To run it, data/Online_Retail.xlsx and data/groceries.csv must sit in a data folder beside this notebook.

In [ ]:
# mlxtend has Apriori, FP-Growth, and the TransactionEncoder; openpyxl reads the xlsx
%pip install mlxtend openpyxl

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import mlxtend
import pandas as pd

from common import (ALGOS, MEMORY_CAP_GB, MINSUP_GRID, TIMEOUT_S, encode,
                    load_groceries, load_online_retail, profile, run_grid)

NOTEBOOK_START = time.perf_counter()

# we record versions because benchmark numbers depend on them
print("pandas", pd.__version__, "| mlxtend", mlxtend.__version__)

os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)

Task 3.1: Data Preprocessing

Transform raw transactional logs into binary one-hot transaction matrices (Pandas TransactionEncoder or sparse matrices). Handle missing item IDs, line cancellations, and non-product noise.

In [ ]:
# we clean Online Retail step by step and log how many rows and invoices survive each step
retail_tx, retail_labels, cleaning_log = load_online_retail()
cleaning_log

In [ ]:
# we spot-check the result: counts, and one basket shown with readable labels
print("baskets:", len(retail_tx))
print("distinct products:", len(retail_labels))
print("example basket:", [retail_labels[c] for c in retail_tx[0][:5]])

In [ ]:
# Groceries is already one basket per line, so loading is all it needs
groceries_tx = load_groceries()
print("baskets:", len(groceries_tx))
print("example basket:", groceries_tx[0])

In [ ]:
# we one-hot encode both datasets with TransactionEncoder and compare their shape and density
X_groc = encode(groceries_tx)
X_retail = encode(retail_tx)

pd.DataFrame([profile("Groceries", X_groc), profile("Online Retail", X_retail)])

In [ ]:
# the most common items show how skewed each dataset is (Person B uses this for cross-support)
top_groc = X_groc.mean().sort_values(ascending=False).head(10).round(3)
top_retail = X_retail.mean().sort_values(ascending=False).head(10).round(3)
top_retail.index = [retail_labels[c] for c in top_retail.index]

print("Groceries, top 10 item supports"); print(top_groc.to_string())
print("\nOnline Retail, top 10 item supports"); print(top_retail.to_string())

For Report (to write after the run): which rows each cleaning step removed and why, why invoices with no CustomerID were dropped, why items are keyed by StockCode with the most common description as the label, and how the density difference between the two datasets should affect mining.

Task 3.2: Apriori vs. FP-Growth

Implement or configure Apriori and FP-Growth algorithms. Benchmark execution runtime (seconds) and peak memory usage across minsup thresholds ranging from 0.001 to 0.05.

We benchmark three settings across six minsup values: mlxtend's default Apriori, Apriori with low_memory set to True, and FP-Growth. Runtime comes from time.perf_counter on runs that are not memory traced. Runs under 60 seconds repeat three times and we report the median. Peak memory comes from one separate run under tracemalloc, because tracing slows Python down. A run past five minutes is recorded as a DNF. Default Apriori is skipped when its two-item candidate arrays would need more than four gigabytes, because on Online Retail at minsup 0.01 it asks for about eleven gigabytes and the kernel dies. Once a setting fails, every lower minsup is marked as skipped, because lowering minsup can only add itemsets. All runs for the report come from one machine.

In [ ]:
# Groceries is small, so this cell takes well under a minute
bench_groc = run_grid(X_groc, "Groceries")
bench_groc

In [ ]:
# Online Retail takes several minutes; the low-memory Apriori runs at the lowest minsup are the slow part
bench_retail = run_grid(X_retail, "Online Retail")
bench_retail

In [ ]:
# we save the results so the report never depends on re-running the benchmark
bench = pd.concat([bench_groc, bench_retail], ignore_index=True)
bench.to_csv("results/benchmark.csv", index=False)
print("saved results/benchmark.csv")

In [ ]:
# plot layout reused from Assignment 2 (final_submission.ipynb cell 24): one line per setting, labelled axes, legend
def plot_benchmark(bench, column, ylabel, filename):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, dataset in zip(axes, ["Groceries", "Online Retail"]):
        for algo in ALGOS:
            rows = bench[(bench["dataset"] == dataset) & (bench["algo"] == algo) & bench[column].notna()]
            ax.plot(rows["min_support"], rows[column], marker="o", label=algo)
        ax.set_xscale("log")
        ax.set_yscale("log")
        # we label the ticks with the minsup values we actually ran instead of powers of ten
        ax.set_xticks(MINSUP_GRID, [str(m) for m in MINSUP_GRID])
        ax.xaxis.set_minor_locator(plt.NullLocator())
        ax.invert_xaxis()   # lower minsup = harder, so difficulty grows to the right
        ax.set_xlabel("minimum support (log scale)")
        ax.set_ylabel(ylabel)
        ax.set_title(dataset)
        ax.grid(True, which="both", alpha=0.3)
        ax.legend()
    fig.tight_layout()
    fig.savefig(f"figures/{filename}", dpi=150)
    plt.show()

plot_benchmark(bench, "runtime_s", "runtime (s, log scale)", "benchmark_runtime.png")

In [ ]:
plot_benchmark(bench, "peak_mb", "peak traced memory (MB, log scale)", "benchmark_memory.png")

In [ ]:
# a line that stops early means the next setting failed; this table says how
bench[bench["status"] != "ok"][["dataset", "algo", "min_support", "status", "runtime_s"]]

For Report (to write after the run): where FP-Growth overtakes Apriori on each dataset, why default Apriori runs out of memory on Online Retail but not on Groceries, what low_memory trades away, and how the number and length of frequent itemsets grow as minsup drops.

Part II: AI Integration Protocol and Prompt Logging

Prompt an AI tool to generate code for candidate rule generation using Apriori. In your report, evaluate whether the AI correctly applied anti-monotone candidate pruning or blindly evaluated all 3^d - 2^(d+1) + 1 possible rules.

Prompt to use, copied exactly into the AI tool with nothing added, so the audit tests what the AI does by default: Write Python code that takes a list of transactions and generates candidate association rules using the Apriori algorithm, given a minimum support and a minimum confidence.

The full exchange, with screenshots, goes in the Word document. The AI's code goes unedited in the raw cell below, then the audit, then our fixed version, the same layout as the Assignment 2 audit.

Audit (to write after reading the AI code). We answer four questions. First, are the size k candidates built only by joining frequent itemsets of size k minus 1? Second, is a candidate dropped when any of its size k minus 1 subsets is infrequent? This is the prune step in apriori-gen from Agrawal and Srikant (1994), section 2.1.1, and it is what mlxtend's own generate_new_combinations does. Third, during rule generation, does the code use the fact that confidence can only fall as items move from the antecedent to the consequent, so it stops growing a consequent once a rule fails? Fourth, does it instead loop over every subset of every itemset, or over all items, which is the brute force the spec warns about? We then check the AI's frequent itemsets against mlxtend on Groceries.

In [ ]:
# how many rules brute force would have to check: R = 3^d - 2^(d+1) + 1 for d items
def total_possible_rules(d):
    return 3**d - 2**(d + 1) + 1

# Python integers never overflow, but 3^3659 is too big for a float, so large counts print as a power of ten
for name, d in [("textbook example", 6), ("Groceries", X_groc.shape[1]), ("Online Retail", X_retail.shape[1])]:
    r = total_possible_rules(d)
    shown = r if r < 10**6 else f"about 10^{len(str(r)) - 1}"
    print(f"{name:<17} d={d:<5} rules = {shown}")

In [ ]:
# correctness check: the AI's frequent itemsets must match mlxtend's exactly on the same data
from common import mine

reference = mine(X_groc, "fpgrowth", 0.01)
reference_sets = set(reference["itemsets"])
print("mlxtend frequent itemsets at minsup 0.01:", len(reference_sets))

# after fixing the AI code, put its itemsets in ai_sets as a set of frozensets, then run:
# print("missing from AI:", len(reference_sets - ai_sets), "| extra in AI:", len(ai_sets - reference_sets))

In [ ]:
print(f"Total notebook runtime: {(time.perf_counter() - NOTEBOOK_START) / 60:.1f} minutes")